# Joint neuron/adapter audit — Colab rehearsal (bug-finding only)

Runs the real pipeline on a Colab GPU **as a bounded rehearsal** (small sizes: 2+2 FP64 rows, 4/4 calibration rows, two benchmark budgets, a 1 h 03b ceiling, 1 batch per arm in 04): restore the sealed preparation snapshot, then 03b qualification (`--rehearsal`), sign-off, a 1-batch-per-arm 04 production run, and 05 selection in prototype mode.
Every output is labelled REHEARSAL and **can never feed the real Pod run**. Production code refuses it.

**Runtime:** GPU **A100** recommended, with High-RAM if available. The FP64 noise-floor stage needs about 13 GB for an FP64 copy of the model. A T4 will fall back to CPU FP64, which can exhaust standard RAM.
**Secret:** add `HF_TOKEN` in Colab *Secrets* (key icon) with read/write access to `Prasadmahadik/spar-sampling-sft-artifacts`.

If a step fails, run the **last cell** anyway. It uploads all rehearsal evidence so the error can be diagnosed.

## 1. Settings

In [ ]:
BRANCH = "neuron-audit-phase1"
REPO_URL = "https://github.com/mahadikprasad15/spar-pilot.git"
WORK = "/content/audit"                      # everything lives under here (local Colab disk)
RUN_TAG = "r1"                               # bump for a fresh rehearsal (r2, r3, ...)
MAX_BATCHES_PER_ARM = 1                      # rehearsal 04 limit
REHEARSAL_CAP_HOURS = 2

## 2. Clone the branch and install the pinned packages

Steps run as subprocesses, so the notebook kernel never imports torch and no restart is needed.

In [ ]:
import os, subprocess, sys, json, pathlib
def sh(cmd, env=None, check=True):
    """Run a shell step and stream its stdout+stderr into the cell (Colab hides a child's own stderr)."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"command failed ({code}): {cmd}")
    return subprocess.CompletedProcess(cmd, code)

os.makedirs(WORK, exist_ok=True)
REPO = f"{WORK}/spar-pilot"
if not os.path.isdir(REPO):
    sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO}")
else:
    sh(f"git -C {REPO} fetch origin {BRANCH} && git -C {REPO} checkout {BRANCH} && git -C {REPO} pull --ff-only")
COMMIT = subprocess.run(f"git -C {REPO} rev-parse HEAD", shell=True, text=True, capture_output=True).stdout.strip()
print("commit", COMMIT)
sh(f"{sys.executable} -m pip install -q torch==2.9.0 transformers==4.57.6 peft==0.18.0 accelerate==1.12.0 "
   "huggingface-hub==0.36.0 tokenizers==0.22.1 safetensors pyyaml pyarrow jinja2 matplotlib")
sh(f"{sys.executable} -c 'import torch, peft, transformers; print(torch.__version__, peft.__version__, transformers.__version__, torch.cuda.get_device_name(0))'")

## 3. Environment (secrets never printed)

In [ ]:
from google.colab import userdata
ENV = dict(os.environ)
ENV.update(HF_TOKEN=userdata.get("HF_TOKEN"), HF_HOME=f"{WORK}/hf", PYTHONUNBUFFERED="1",
           CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONPATH=REPO, HF_HUB_ENABLE_HF_TRANSFER="0")
ONLINE = dict(ENV, HF_HUB_OFFLINE="0", TRANSFORMERS_OFFLINE="0")
print("HF token set:", bool(ENV["HF_TOKEN"]))
PY = sys.executable

## 4. Restore the sealed preparation snapshot (ticket 07 restore)

Contents: inventory-v2, sequences-v2, the six step-64 adapters and the tokenizer, all hash-verified.

In [ ]:
PREP = f"{WORK}/prep"
RECEIPT = f"{REPO}/.scratch/joint-neuron-adapter-audit/prep-v2.hf-upload.json"
if not os.path.isdir(f"{PREP}/runs"):
    sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_export restore --upload_receipt {RECEIPT} --clean_root {PREP}", env=ONLINE)
sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_sequences verify --run_dir {PREP}/runs/joint-neuron-adapter-audit/sequences/sequences-v2", env=ENV)

## 5. Download the pinned base model (online) and check its file hashes

In [ ]:
inventory = json.loads(pathlib.Path(f"{PREP}/runs/joint-neuron-adapter-audit/inventory/inventory-v2/inventory.json").read_text())
base = inventory["base"]
request = dict(model=base["model"], revision=base["model_revision"], expected=base["expected_file_sha256"])
pathlib.Path(f"{WORK}/model_request.json").write_text(json.dumps(request))
script = """
import hashlib, json, pathlib, sys
from huggingface_hub import snapshot_download
request = json.loads(pathlib.Path(sys.argv[1]).read_text())
path = snapshot_download(request["model"], revision=request["revision"])
bad = []
for name, digest in request["expected"].items():
    h = hashlib.sha256()
    with open(pathlib.Path(path) / name, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        bad.append(name)
size = sum(p.stat().st_size for p in pathlib.Path(path).rglob("*") if p.is_file())
print(json.dumps(dict(path=path, mismatched=bad, bytes=size)))
"""
out = subprocess.run([PY, "-c", script, f"{WORK}/model_request.json"], env=ONLINE, text=True, capture_output=True)
if out.returncode:
    print(out.stderr[-3000:]); raise RuntimeError("model download failed")
MODEL = json.loads(out.stdout.strip().splitlines()[-1]); print(MODEL)
assert not MODEL["mismatched"], "base model files differ from the pinned hashes"


## 6. Rehearsal plan and worker config

In [ ]:
PLAN = f"{REPO}/.scratch/joint-neuron-adapter-audit/frozen_plan_v1.yaml"
REHEARSAL_PLAN = f"{WORK}/rehearsal_plan.yaml"
sh(f"cd {REPO} && {PY} -c \"from pilot_eval import neuron_audit_plan as p; p.make_rehearsal_plan('{PLAN}', '{REHEARSAL_PLAN}', small=True)\"", env=ENV)
adapter_dirs = {arm: f"{PREP}/{rec['checkpoints']['64']['path']}"
                for arm, rec in inventory["arms"].items() if rec["checkpoints"].get("64")}
for d in [f"{WORK}/out", f"{WORK}/volume", f"{WORK}/tmp", f"{WORK}/local"]:
    os.makedirs(d, exist_ok=True)
CONFIG = dict(artifact_root=PREP, output_root=f"{WORK}/out", run_id=f"rehearsal-q-{RUN_TAG}",
              volume_dir=f"{WORK}/volume", local_temp=f"{WORK}/tmp", model_dir=MODEL["path"],
              tokenizer_dir=f"{PREP}/inputs/models/{base['tokenizer_revision']}", reference_arm="gold-sft",
              adapter_dirs=adapter_dirs, model_cache_bytes=MODEL["bytes"], commit=COMMIT)
pathlib.Path(f"{WORK}/qualify.config.json").write_text(json.dumps(CONFIG, indent=2))
print(json.dumps({k: v for k, v in CONFIG.items() if k != "adapter_dirs"}, indent=2)); print(sorted(adapter_dirs))

## 7. Ticket 03b qualification (rehearsal)

Stages: preflight → masks → arms → benchmark → batch agreement → FP64 floor → BF16 diagnostic → batch freeze. It resumes from sealed stages if re-run. Progress: `out/runs/joint-neuron-adapter-audit/qualification/<run_id>/logs/progress.jsonl`.

In [ ]:
sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_qualify run --rehearsal --plan {REHEARSAL_PLAN} --config {WORK}/qualify.config.json", env=ENV)
QUAL = f"{WORK}/out/runs/joint-neuron-adapter-audit/qualification/{CONFIG['run_id']}"
print(pathlib.Path(f"{QUAL}/qualification.json").read_text())

## 8. Review flags and sign-off (rehearsal)

Look at the raised flags and the discrepancy-vs-layer plot, then record the sign-off (it is a rehearsal, so `accept` here only exercises the flow).

In [ ]:
result = json.loads(pathlib.Path(f"{QUAL}/stages/batch_agreement/result.json").read_text())
print("raised flags:", [f["flag"] for f in result["review_flags"]], "| logits max |diff|:", result["logits_max_abs"])
png = pathlib.Path(f"{QUAL}/stages/batch_agreement/discrepancy_vs_layer.png")
if png.exists():
    from IPython.display import Image, display; display(Image(str(png)))
if not pathlib.Path(f"{QUAL}/review/signoff.json").exists():
    sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_qualify signoff --run_dir {QUAL} --reviewer colab-rehearsal --decision accept --notes 'rehearsal only'", env=ENV)

## 9. Ticket 04 production (rehearsal, MAX_BATCHES_PER_ARM batches per arm)

Uploads one small verified package per arm under `neuron-audit/rehearsal-colab-<tag>/` plus a reference-only final manifest. The Pod-stop command is not configured here.

In [ ]:
PCONFIG = dict(CONFIG, run_id=f"rehearsal-p-{RUN_TAG}", local_root=f"{WORK}/local", volume_root=f"{WORK}/volume",
               qualification_root=f"{WORK}/out", qualification_run_id=CONFIG["run_id"], label=f"colab-{RUN_TAG}",
               max_batches_per_arm=MAX_BATCHES_PER_ARM, rehearsal_cap_hours=REHEARSAL_CAP_HOURS)
pathlib.Path(f"{WORK}/production.config.json").write_text(json.dumps(PCONFIG, indent=2))
sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_production --rehearsal --plan {REHEARSAL_PLAN} --config {WORK}/production.config.json", env=ENV)

## 10. Ticket 05 selection (prototype over the rows every arm collected)

In [ ]:
STORES = f"{WORK}/volume/runs/joint-neuron-adapter-audit/production/{PCONFIG['run_id']}/blocks"
sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_selection --rehearsal --plan {REHEARSAL_PLAN} --stores {STORES} "
   f"--sequences_run {PREP}/runs/joint-neuron-adapter-audit/sequences/sequences-v2 --output_root {WORK}/out "
   f"--run_id rehearsal-s-{RUN_TAG} --layers 0-27 --qualification_dir {QUAL}", env=ENV)

## 10b. Ticket 06 report (prototype)

Held-out estimates for the frozen rehearsal lists; labelled PROTOTYPE / REHEARSAL. Output: `out/runs/joint-neuron-adapter-audit/report/rehearsal-report-<tag>/report.md`.

In [ ]:
SEL = f"{WORK}/out/runs/joint-neuron-adapter-audit/selection/rehearsal-s-{RUN_TAG}"
sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_report --rehearsal --plan {REHEARSAL_PLAN} --selection_dir {SEL} "
   f"--stores {STORES} --sequences_run {PREP}/runs/joint-neuron-adapter-audit/sequences/sequences-v2 "
   f"--qualification_dir {QUAL} --output_root {WORK}/out --run_id rehearsal-report-{RUN_TAG} --layers 0-27", env=ENV)
print(pathlib.Path(f"{WORK}/out/runs/joint-neuron-adapter-audit/report/rehearsal-report-{RUN_TAG}/report.md").read_text()[:4000])


## 11. Upload rehearsal evidence for diagnosis (run this even after a failure)

Snapshots the qualification, production state and selection outputs (sealed parts plus logs/status) and uploads them under `neuron-audit/rehearsal-evidence-<tag>/`. Paste the printed receipt back.

In [ ]:
request = dict(work=WORK, tag=RUN_TAG, repo="Prasadmahadik/spar-sampling-sft-artifacts")
pathlib.Path(f"{WORK}/evidence_request.json").write_text(json.dumps(request))
for d in ["production", "selection"]:  # production state/marker live on the stand-in volume; copy them under out/
    src = pathlib.Path(f"{WORK}/volume/runs/joint-neuron-adapter-audit/{d}")
    if src.exists():
        sh(f"mkdir -p {WORK}/out/runs/joint-neuron-adapter-audit/{d}-meta && cp -r {src}/*/meta {WORK}/out/runs/joint-neuron-adapter-audit/{d}-meta/", check=False)
script = """
import json, pathlib, sys
from pilot_eval import neuron_audit_export as ex
r = json.loads(pathlib.Path(sys.argv[1]).read_text())
s = ex.snapshot(r["work"] + "/out", ["runs/joint-neuron-adapter-audit"], "rehearsal-evidence-" + r["tag"],
                stage="rehearsal", include_unsealed_evidence=True)
print("completeness", s["completeness"], "files", len(s["files"]), "unsealed", len(s["unsealed"]))
up = ex.upload(r["work"] + "/out/" + s["snapshot_path"], r["work"] + "/out", repo=r["repo"], temp_dir=r["work"] + "/tmp")
print("PASTE THIS BACK:", json.dumps(dict(repo=up["repo"], revision=up["revision"], archive=up["archive"])))
"""
subprocess.run([PY, "-c", script, f"{WORK}/evidence_request.json"], env=ONLINE, cwd=REPO, text=True)
